In [1]:
import pandas as pd

## Load quora dataset

In [2]:
quora_data = pd.read_csv('../data/train.csv')

In [3]:
quora_data.head(5)

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [5]:
quora_q1_lst = quora_data['question1'].tolist()
quora_q2_lst = quora_data['question2'].tolist()

## Clean up text

In [27]:
def cleanup(text):
    try:
        text = text.lower()
        text = text.replace('?', ' ?')
    except AttributeError:
        print("AttributeError: text: {}".format(text))
    return text

In [29]:
cleaned_quora_q1_lst = map(cleanup, quora_q1_lst)
cleaned_quora_q2_lst = map(cleanup, quora_q2_lst)

## Load Wiki questions/sentences

In [16]:
wiki_data = pd.read_csv('../data/WikiQACorpus/WikiQA.tsv', delimiter='\t')

In [17]:
wiki_data.head(5)

,QuestionID,Question,DocumentID,DocumentTitle,SentenceID,Sentence,Label
0,Q0,HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US,D0,African immigration to the United States,D0-0,African immigration to the United States refer...,0
1,Q0,HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US,D0,African immigration to the United States,D0-1,The term African in the scope of this article ...,0
2,Q0,HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US,D0,African immigration to the United States,D0-2,From the Immigration and Nationality Act of 19...,0
3,Q0,HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US,D0,African immigration to the United States,D0-3,African immigrants in the United States come f...,0
4,Q0,HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US,D0,African immigration to the United States,D0-4,"They include people from different national, l...",0


In [18]:
wiki_q_lst = wiki_data['Question'].tolist()
wiki_sent_lst = wiki_data['Sentence'].tolist()

In [19]:
print("Size of Quora dataset: {}".format(quora_data.shape[0]))
print("Size of Wiki dataset: {}".format(wiki_data.shape[0]))

Size of Quora dataset: 404290
Size of Wiki dataset: 29208


In [30]:
cleaned_wiki_q_lst = map(cleanup, wiki_q_lst)
cleaned_wiki_sent_lst = map(cleanup, wiki_sent_lst)

In [23]:
list(cleaned_wiki_q_lst)[0]

'how african americans were immigrated to the us'

In [24]:
wiki_q_lst[0]

'HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US'

In [25]:
def convert_list_to_list_of_dicst(lst, label='__label__question'):
    new_lst = []
    for item in lst:
        new_lst.append({'label': label, 'text': item})
    return new_lst

In [31]:
quora_q1_lst_of_dcts = convert_list_to_list_of_dicst(cleaned_quora_q1_lst, '__label__question')
quora_q2_lst_of_dcts = convert_list_to_list_of_dicst(cleaned_quora_q2_lst, '__label__question')

cleaned_wiki_sent_lst_of_dcts = convert_list_to_list_of_dicst(cleaned_wiki_sent_lst, '__label__not-question')
cleaned_wiki_q_lst_of_dcts = convert_list_to_list_of_dicst(cleaned_wiki_q_lst, '__label__question')

AttributeError: text: nan
AttributeError: text: nan
AttributeError: text: nan


In [36]:
cleaned_wiki_q_lst_of_dcts[100]

{'label': '__label__question', 'text': 'how big do sebaceous cysts get'}

In [37]:
final_lst_of_dcts = quora_q1_lst_of_dcts + quora_q2_lst_of_dcts + cleaned_wiki_sent_lst_of_dcts + cleaned_wiki_q_lst_of_dcts


In [38]:
assert len(final_lst_of_dcts) == len(quora_q1_lst_of_dcts) + len(quora_q2_lst_of_dcts) + len(cleaned_wiki_sent_lst_of_dcts) + len(cleaned_wiki_q_lst_of_dcts)

In [39]:
dataset = pd.DataFrame(final_lst_of_dcts)

## Randomly shuffle pandas dataframe

In [50]:
dataset = dataset.sample(frac=1)

In [51]:
dataset.head()

,label,text
302877,__label__question,which cycle has the greater thermal efficiency...
545410,__label__question,what is a good way to compose music ?
403363,__label__question,are there any other websites like quora ?
40310,__label__question,how do i track someone using a 800 number ?
334551,__label__question,what is the difference between row and tuple i...


## Write dataset to disk for fastetxt modeling

In [52]:
dataset.to_csv('../data/quora_wiki.txt', sep=' ', header=False, index=False)

## Split data to train/valid

head -n 693646 data/quora_wiki.txt > data/quora_wiki.train

head -n 173411 data/quora_wiki.txt > data/quora_wiki.valid

693646 data/quora_wiki.train

173411 data/quora_wiki.valid